In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# Carregar os dados
file_path = 'https://raw.githubusercontent.com/CD-CrashTest/Crash_Test/refs/heads/main/scrappingNewDf.csv'  # Substitua pelo caminho do seu arquivo
data = pd.read_csv(file_path)

# Remover 'kg' da coluna 'Kerb Weight' e converter para num?rico
data['Kerb Weight'] = data['Kerb Weight'].str.replace('kg', '').astype(float)

data.head()

In [ ]:
# Preencher valores ausentes nas colunas num?ricas com valores aleat?rios entre m?dia ? desvio padr?o
numeric_cols = data.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    mean = data[col].mean()
    std_dev = data[col].std()
    data[col] = data[col].apply(lambda x: np.random.uniform(mean - std_dev, mean + std_dev) if pd.isna(x) else x)

In [ ]:
# Separar as features e a vari?vel alvo
features = data.select_dtypes(include=[np.number]).drop(columns=['rating'], errors='ignore')
target = data['rating']

In [ ]:
# Dividir os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
# Inicializar e treinar o classificador Random Forest
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# Fazer previs?es e avaliar o modelo
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

In [ ]:
# Exibir resultados
print("Acur?cia:", accuracy)
print("Relat?rio de Classifica??o:\n", report)

In [ ]:
# Kerb Weight
# Safety Features
# CRS Installation Check
# Class
# Seatbelt Reminder

# Selecionar as features relevantes
features_selected = ['Kerb Weight', 'Safety Features', 'CRS Installation Check', 'Class', 'Seatbelt Reminder']
features = data[features_selected]

# Converter features categ?ricas para num?ricas usando one-hot encoding
categorical_features = ['Class', 'Safety Features', 'CRS Installation Check', 'Seatbelt Reminder'] #Especificar as colunas categoricas
numerical_features = ['Kerb Weight'] #Especificar as colunas numericas

# Converter as features categ?ricas para num?ricas usando one-hot encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_features = encoder.fit_transform(features[categorical_features])

# Criar um novo dataframe com as features codificadas
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_features))

# Concatenar as features num?ricas com as codificadas
X = pd.concat([features[numerical_features], encoded_df], axis=1)

# Converter o rating para num?rico (se necess?rio)
y = data['rating'].astype(int)

# Dividir os dados em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar os dados num?ricos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Criar o modelo
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(6, activation='softmax')) # 6 classes (0 a 5)

# Compilar o modelo
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Treinar o modelo
model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Avaliar o modelo
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f'Acur?cia do modelo: {accuracy}')

In [ ]:
# Solicitar ao usu?rio os valores das features
kerb_weight = float(input("Digite o peso do ve?culo em Kg: "))
safety_features = input("Digite nota de seguran?a (0-10): ")
crs_installation = input("Digite a nota de facilidade de instala??o de cadeirinha infantil (0-12): ")
vehicle_class = input("Digite a classe do ve?culo: ")
seatbelt_reminder = input("Ve?culo poss?i alerta de cinto de seguran?a (sim ou n?o): ")

# Criar um DataFrame com os valores de entrada do usu?rio
user_input = pd.DataFrame({
    'Kerb Weight': [kerb_weight],
    'Safety Features': [safety_features],
    'CRS Installation Check': [crs_installation],
    'Class': [vehicle_class],
    'Seatbelt Reminder': [seatbelt_reminder]
})

# Pr?-processar os dados de entrada do usu?rio da mesma forma que os dados de treinamento
categorical_features = ['Class', 'Safety Features', 'CRS Installation Check', 'Seatbelt Reminder']
numerical_features = ['Kerb Weight']

# Converter as features categ?ricas para num?ricas usando one-hot encoding (usando o mesmo encoder treinado antes)
encoded_features = encoder.transform(user_input[categorical_features])
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_features))

# Concatenar as features num?ricas com as codificadas
X_user = pd.concat([user_input[numerical_features], encoded_df], axis=1)

# Normalizar os dados de entrada do usu?rio usando o mesmo scaler usado no treino
X_user_scaled = scaler.transform(X_user)

# Fazer a previs?o com o modelo treinado
prediction = model.predict(X_user_scaled)

# Obter a classe prevista (?ndice da probabilidade m?xima)
predicted_class = np.argmax(prediction)

print(f"Previs?o de classifica??o (Rating): {predicted_class}")